In [2]:
# =============================================================================
# TP53 MUTANT DOCKING BENCHMARK – SMALL-MOLECULE LIGAND (NUTLIN-3A)
# =============================================================================
# This script:
#   1. Downloads Nutlin-3a (a known TP53-binding small molecule) from PubChem
#   2. Converts it to a valid PDBQT using OpenBabel (simple and reliable)
#   3. Runs Vina docking for all your mutant PDBQT receptors
#   4. Generates ranked results with statistics, including log files
#
# INPUT: Your mutant PDBQT files (upload only these!)
# OUTPUT: docking_results.csv, docking_results.xlsx, ZIP archive with all files
# =============================================================================

# -----------------------------------------------------------------------------
# 1. Install dependencies
# -----------------------------------------------------------------------------
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y autodock-vina openbabel > /dev/null 2>&1
!pip install -q pandas openpyxl tqdm biopython

import os
import glob
import zipfile
import subprocess
import pandas as pd
import numpy as np
import re
import urllib.request
from tqdm import tqdm
from google.colab import files
from Bio.PDB import PDBParser

# -----------------------------------------------------------------------------
# 2. Create directories
# -----------------------------------------------------------------------------
RECEPTOR_DIR = "/content/receptors"
LIGAND_DIR = "/content/ligand"
RESULTS_DIR = "/content/docking_results"
os.makedirs(RECEPTOR_DIR, exist_ok=True)
os.makedirs(LIGAND_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"📁 Receptor directory: {RECEPTOR_DIR}")
print(f"📁 Ligand directory: {LIGAND_DIR}")
print(f"📁 Results directory: {RESULTS_DIR}")

# -----------------------------------------------------------------------------
# 3. Upload mutant receptor PDBQT files (ONLY THESE!)
# -----------------------------------------------------------------------------
print("\n📤 Upload your mutant PDBQT files (or a ZIP archive).")
print("   The small-molecule ligand will be downloaded automatically.")
uploaded_receptors = files.upload()

for fn, content in uploaded_receptors.items():
    filepath = os.path.join(RECEPTOR_DIR, fn)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"✅ Saved receptor: {fn}")

# Extract ZIP archives
zip_files = [f for f in os.listdir(RECEPTOR_DIR) if f.endswith('.zip')]
for zip_name in zip_files:
    zip_path = os.path.join(RECEPTOR_DIR, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(RECEPTOR_DIR)
    print(f"📦 Extracted: {zip_name}")
    os.remove(zip_path)

receptor_files = glob.glob(os.path.join(RECEPTOR_DIR, '*.pdbqt'))
if not receptor_files:
    raise SystemExit("❌ No receptor PDBQT files found.")

print(f"\n🔍 Found {len(receptor_files)} receptor files.")

# -----------------------------------------------------------------------------
# 4. Download and prepare small-molecule ligand (Nutlin-3a) using OpenBabel
# -----------------------------------------------------------------------------
print("\n🧪 Downloading Nutlin-3a (PubChem CID: 216239) as the ligand...")

# Download Nutlin-3a SDF from PubChem
pubchem_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/216239/record/SDF/?record_type=3d&response_type=display"
sdf_path = os.path.join(LIGAND_DIR, "nutlin3a.sdf")
urllib.request.urlretrieve(pubchem_url, sdf_path)
print(f"✅ Downloaded Nutlin-3a SDF ({os.path.getsize(sdf_path)} bytes)")

# Convert SDF to PDB (optional intermediate, but not strictly needed)
print("🔄 Converting SDF to PDB...")
pdb_path = os.path.join(LIGAND_DIR, "nutlin3a.pdb")
subprocess.run(['obabel', sdf_path, '-O', pdb_path], capture_output=True, text=True)

# Convert directly to PDBQT with hydrogens and charges
print("🔄 Converting to PDBQT with OpenBabel...")
ligand_pdbqt = os.path.join(LIGAND_DIR, "ligand_strict.pdbqt")

# Run OpenBabel: add hydrogens (-h), assign Gasteiger charges at pH 7.4 (-p 7.4)
cmd = ['obabel', pdb_path, '-O', ligand_pdbqt, '-h', '-p', '7.4']
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ OpenBabel error: {result.stderr}")
    raise SystemExit("Ligand preparation failed.")

# Ensure the PDBQT has ROOT/ENDROOT/TORSDOF (Vina requires these)
with open(ligand_pdbqt, 'r') as f:
    lines = f.readlines()

if not any(line.startswith('ROOT') for line in lines):
    atom_lines = [l for l in lines if l.startswith(('ATOM', 'HETATM'))]
    if atom_lines:
        with open(ligand_pdbqt, 'w') as f:
            f.write('ROOT\n')
            f.writelines(atom_lines)
            f.write('ENDROOT\n')
            f.write('TORSDOF 0\n')
        print("✅ Manually added ROOT/ENDROOT/TORSDOF.")

# Verify ligand
with open(ligand_pdbqt, 'r') as f:
    content = f.read()
    atom_count = content.count('ATOM') + content.count('HETATM')
    has_root = 'ROOT' in content
    print(f"🔎 Ligand atoms: {atom_count}, ROOT present: {has_root}")

if atom_count == 0 or not has_root:
    raise SystemExit("❌ Ligand preparation failed. Please check manually.")

ligand_path = ligand_pdbqt
print(f"✅ Ligand ready: {os.path.basename(ligand_path)}")

# -----------------------------------------------------------------------------
# 5. Calculate grid center from first receptor (robust)
# -----------------------------------------------------------------------------
def calculate_center(pdbqt_path):
    coords = []
    # Try Biopython first
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure('protein', pdbqt_path)
        ca_atoms = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    for atom in residue:
                        if atom.get_name() == 'CA':
                            ca_atoms.append(atom.get_coord())
        if ca_atoms:
            center = np.mean(ca_atoms, axis=0)
            return center[0], center[1], center[2]
    except Exception:
        pass

    # Fallback: parse coordinates from PDBQT directly (columns 30-54)
    with open(pdbqt_path, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append((x, y, z))
                except:
                    continue
    if not coords:
        raise ValueError(f"No coordinates found in {pdbqt_path}")
    center = np.mean(coords, axis=0)
    return center[0], center[1], center[2]

first_receptor = receptor_files[0]
center_x, center_y, center_z = calculate_center(first_receptor)
print(f"🎯 Grid center: ({center_x:.3f}, {center_y:.3f}, {center_z:.3f})")

# -----------------------------------------------------------------------------
# 6. Docking function – MAXIMUM ACCURACY with robust score parsing
# -----------------------------------------------------------------------------
def run_vina_docking(receptor_path, ligand_path, center, out_dir,
                     size_x=30, size_y=30, size_z=30,
                     exhaustiveness=20, num_modes=20, energy_range=2):
    base = os.path.splitext(os.path.basename(receptor_path))[0]
    out_file = os.path.join(out_dir, f"{base}_docked.pdbqt")
    log_file = os.path.join(out_dir, f"{base}_log.txt")

    config = f"""receptor = {receptor_path}
ligand = {ligand_path}
center_x = {center[0]:.3f}
center_y = {center[1]:.3f}
center_z = {center[2]:.3f}
size_x = {size_x}
size_y = {size_y}
size_z = {size_z}
exhaustiveness = {exhaustiveness}
num_modes = {num_modes}
energy_range = {energy_range}
out = {out_file}
"""
    config_path = "/tmp/config.txt"
    with open(config_path, 'w') as f:
        f.write(config)

    try:
        with open(log_file, 'w') as log_f:
            result = subprocess.run(['vina', '--config', config_path],
                                    capture_output=True, text=True, timeout=600)
            log_f.write(result.stdout)
            log_f.write(result.stderr)
    except subprocess.TimeoutExpired:
        print(f"⏰ Timeout for {base}")
        return None, None

    if result.returncode != 0:
        if "ERROR" in result.stderr or result.stderr.strip() != "":
            print(f"❌ Vina error for {base}: {result.stderr[:200]}")
            return None, None

    # ----- ROBUST SCORE PARSING -----
    score = None
    # 1. Try to parse from stdout (mode 1 score line)
    for line in result.stdout.splitlines():
        match = re.match(r'\s*1\s+([-+]?\d+\.\d+)', line)
        if match:
            score = float(match.group(1))
            break

    # 2. If not found, try the log file
    if score is None and os.path.exists(log_file):
        with open(log_file, 'r') as f:
            log_content = f.read()
            # Look for "REMARK VINA RESULT:" line
            for line in log_content.splitlines():
                if "REMARK VINA RESULT:" in line:
                    parts = line.split()
                    if len(parts) >= 4:
                        try:
                            score = float(parts[3])
                            break
                        except:
                            pass
            # If still none, use regex for mode 1
            if score is None:
                for line in log_content.splitlines():
                    match = re.match(r'\s*1\s+([-+]?\d+\.\d+)', line)
                    if match:
                        score = float(match.group(1))
                        break

    # 3. Last resort: read the output PDBQT file
    if score is None and os.path.exists(out_file):
        with open(out_file, 'r') as f:
            lines = f.readlines()
            for line in reversed(lines[-50:]):  # check last 50 lines
                if "REMARK VINA RESULT:" in line:
                    parts = line.split()
                    if len(parts) >= 4:
                        try:
                            score = float(parts[3])
                            break
                        except:
                            pass

    if score is None:
        print(f"⚠️ No score found for {base}")
        return None, None

    return score, out_file, log_file   # ← now returning log file path

# -----------------------------------------------------------------------------
# 7. Run docking for all receptors
# -----------------------------------------------------------------------------
results = []
print("\n🔄 Running Vina docking with MAXIMUM ACCURACY settings...")
print(f"   Exhaustiveness: 20")
print(f"   Num modes: 20")
print(f"   Ligand: Nutlin-3a (small molecule, Vina-compatible)\n")

for receptor_path in tqdm(receptor_files, desc="Docking"):
    base = os.path.splitext(os.path.basename(receptor_path))[0]
    score, out_file, log_file = run_vina_docking(receptor_path, ligand_path,
                                                 (center_x, center_y, center_z),
                                                 RESULTS_DIR)
    results.append({
        'receptor': base,
        'affinity_kcal_mol': score,
        'output_file': os.path.basename(out_file) if out_file else None,
        'log_file': os.path.basename(log_file) if log_file else None,
        'status': 'success' if score is not None else 'failed'
    })

# -----------------------------------------------------------------------------
# 8. Generate results table
# -----------------------------------------------------------------------------
df = pd.DataFrame(results)
df_sorted = df.sort_values('affinity_kcal_mol', ascending=True, na_position='last')
df_sorted['rank'] = range(1, len(df_sorted)+1)
df_sorted.loc[df_sorted['affinity_kcal_mol'].isna(), 'rank'] = None

# Reorder columns for clarity
df_sorted = df_sorted[['receptor', 'affinity_kcal_mol', 'rank', 'status', 'output_file', 'log_file']]

csv_path = os.path.join(RESULTS_DIR, "docking_results.csv")
excel_path = os.path.join(RESULTS_DIR, "docking_results.xlsx")
df_sorted.to_csv(csv_path, index=False, float_format='%.2f')
print(f"\n📊 CSV saved: {csv_path}")

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_sorted.to_excel(writer, sheet_name='All Results', index=False, float_format='%.2f')
    successful = df_sorted[df_sorted['affinity_kcal_mol'].notna()]
    summary = {
        'Metric': ['Total receptors', 'Successfully docked', 'Mean affinity (kcal/mol)',
                   'Median affinity (kcal/mol)', 'Best affinity (kcal/mol)',
                   'Worst affinity (kcal/mol)', 'Standard deviation'],
        'Value': [
            len(df_sorted),
            len(successful),
            successful['affinity_kcal_mol'].mean() if len(successful) > 0 else None,
            successful['affinity_kcal_mol'].median() if len(successful) > 0 else None,
            successful['affinity_kcal_mol'].min() if len(successful) > 0 else None,
            successful['affinity_kcal_mol'].max() if len(successful) > 0 else None,
            successful['affinity_kcal_mol'].std() if len(successful) > 0 else None
        ]
    }
    pd.DataFrame(summary).to_excel(writer, sheet_name='Summary', index=False)
print(f"📊 Excel saved: {excel_path}")

# -----------------------------------------------------------------------------
# 9. Zip all results (including log files and docked PDBQTs)
# -----------------------------------------------------------------------------
zip_path = "/content/docking_benchmark_results.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all files in RESULTS_DIR
    for file_path in glob.glob(os.path.join(RESULTS_DIR, '*')):
        zipf.write(file_path, os.path.basename(file_path))

print(f"\n📦 Results zipped: {zip_path}")
files.download(zip_path)

print("\n✅ Classical docking benchmark complete!")
if len(successful) > 0:
    print(f"   - {len(successful)} out of {len(df_sorted)} receptors docked successfully.")
    print(f"   - Best affinity: {successful['affinity_kcal_mol'].min():.2f} kcal/mol")
    print(f"   - Mean affinity: {successful['affinity_kcal_mol'].mean():.2f} kcal/mol")
else:
    print("   ⚠️ All dockings failed. Please check your input files.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00
📁 Receptor directory: /content/receptors
📁 Ligand directory: /content/ligand
📁 Results directory: /content/docking_results

📤 Upload your mutant PDBQT files (or a ZIP archive).
   The small-molecule ligand will be downloaded automatically.


Saving TP53_A159V.pdbqt to TP53_A159V.pdbqt
Saving TP53_A161T.pdbqt to TP53_A161T.pdbqt
Saving TP53_C17F.pdbqt to TP53_C17F.pdbqt
Saving TP53_C44F.pdbqt to TP53_C44F.pdbqt
Saving TP53_C102Y.pdbqt to TP53_C102Y.pdbqt
Saving TP53_C135F.pdbqt to TP53_C135F.pdbqt
Saving TP53_C137F.pdbqt to TP53_C137F.pdbqt
Saving TP53_C137Y.pdbqt to TP53_C137Y.pdbqt
Saving TP53_C141Y.pdbqt to TP53_C141Y.pdbqt
Saving TP53_C176F.pdbqt to TP53_C176F.pdbqt
Saving TP53_C176Y.pdbqt to TP53_C176Y.pdbqt
Saving TP53_C199Y.pdbqt to TP53_C199Y.pdbqt
Saving TP53_C238F.pdbqt to TP53_C238F.pdbqt
Saving TP53_C238Y.pdbqt to TP53_C238Y.pdbqt
Saving TP53_C242F.pdbqt to TP53_C242F.pdbqt
Saving TP53_C275Y.pdbqt to TP53_C275Y.pdbqt
Saving TP53_E126K.pdbqt to TP53_E126K.pdbqt
Saving TP53_E153K.pdbqt to TP53_E153K.pdbqt
Saving TP53_E246K.pdbqt to TP53_E246K.pdbqt
Saving TP53_E247K.pdbqt to TP53_E247K.pdbqt
Saving TP53_E285K.pdbqt to TP53_E285K.pdbqt
Saving TP53_E286K.pdbqt to TP53_E286K.pdbqt
Saving TP53_G86S.pdbqt to TP53_G86S.

Docking: 100%|██████████| 150/150 [10:39:19<00:00, 255.73s/it]



📊 CSV saved: /content/docking_results/docking_results.csv
📊 Excel saved: /content/docking_results/docking_results.xlsx

📦 Results zipped: /content/docking_benchmark_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Classical docking benchmark complete!
   - 150 out of 150 receptors docked successfully.
   - Best affinity: -8.80 kcal/mol
   - Mean affinity: -7.41 kcal/mol
